In [1]:
from random import random, randint, sample
from collections import namedtuple
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [6]:
Inversion = namedtuple('Inversion', 'precio cantidad rendim')

inversiones = [
    Inversion(precio=500, cantidad=5,  rendim=0.15),  # A
    Inversion(precio=700, cantidad=6,  rendim=0.05),  # B
    Inversion(precio=300, cantidad=10, rendim=0.03),  # C
    Inversion(precio=800, cantidad=3,  rendim=0.10),  # D
    Inversion(precio=400, cantidad=7,  rendim=0.10),  # E
]

nombres_bonos = ['A', 'B', 'C', 'D', 'E']
capital = 10000

print(f"{'Bono':<6} {'Precio':>8} {'Cantidad':>10} {'Rendimiento':>12}")
print("-" * 40)
for nombre, inv in zip(nombres_bonos, inversiones):
    print(f"{nombre:<6} {inv.precio:>8} {inv.cantidad:>10} {inv.rendim:>12.2f}")
print(f"\nCapital disponible: ${capital:,}")

Bono     Precio   Cantidad  Rendimiento
----------------------------------------
A           500          5         0.15
B           700          6         0.05
C           300         10         0.03
D           800          3         0.10
E           400          7         0.10

Capital disponible: $10,000


In [8]:
# --- FUNCIÓN OBJETIVO ---
def capitalInvertido(individuo):
    """Calcula el capital total invertido por un individuo."""
    return sum(map(lambda x, y: x * y.precio, individuo, inversiones))

def rendimiento(individuo):
    """Función objetivo: rendimiento total del portafolio."""
    return sum(map(lambda x, y: x * y.precio * y.rendim, individuo, inversiones))

# --- REPARACIÓN DE RESTRICCIONES ---
def ajustaCapital(individuo):
    """Repara individuos que exceden el capital eliminando bonos aleatoriamente."""
    ajustado = individuo[:]
    while capitalInvertido(ajustado) > capital:
        pos = randint(0, len(ajustado) - 1)
        if ajustado[pos] > 0:
            ajustado[pos] -= 1
    return ajustado

# --- INICIALIZACIÓN ---
def creaIndividuo(inversiones, capital):
    """Crea un individuo aleatorio que respete el capital disponible."""
    individuo = [0] * len(inversiones)
    while capitalInvertido(individuo) < capital:
        eleccion = randint(0, len(inversiones) - 1)
        individuo[eleccion] += 1
    return ajustaCapital(individuo)

# --- OPERADOR DE CRUZA (crossover de un punto) ---
def cruza(poblacion, posiciones):
    """Cruza dos individuos: toma genes del padre 1 y un segmento del padre 2."""
    L     = len(poblacion[0])
    hijo  = poblacion[posiciones[0]][:]
    inicio = randint(0, L - 1)
    fin    = randint(inicio + 1, L)
    hijo[inicio:fin] = poblacion[posiciones[1]][inicio:fin]
    return ajustaCapital(hijo)

# --- OPERADOR DE MUTACIÓN ---
def muta(individuo, tasaMutacion):
    """Muta genes aleatoriamente según la tasa de mutación dada."""
    mutado = []
    for i in range(len(individuo)):
        if random() > tasaMutacion:
            mutado.append(individuo[i])
        else:
            mutado.append(randint(0, inversiones[i].cantidad))
    return ajustaCapital(mutado)

# --- OPERADOR DE SELECCIÓN + EVOLUCIÓN ---
def evoluciona(poblacion, generaciones):
    """
    Selección proporcional al rango (rank-based):
    los mejores individuos tienen más probabilidad de reproducirse.
    Elitismo: el mejor individuo siempre pasa a la siguiente generación.
    """
    poblacion.sort(key=lambda x: rendimiento(x))
    N            = len(poblacion)
    tasaMutacion = 0.01
    reproduccion = [x for x in range(N) for y in range(x + 1)]
    historial    = []

    for i in range(generaciones):
        padres = sample(reproduccion, 2)
        while padres[0] == padres[1]:
            padres = sample(reproduccion, 2)

        hijos = [cruza(poblacion, padres) for x in range(N - 1)]
        hijos = [muta(x, tasaMutacion) for x in hijos]
        hijos.append(poblacion[-1])   # elitismo
        poblacion = hijos
        poblacion.sort(key=lambda x: rendimiento(x))
        historial.append(rendimiento(poblacion[-1]))

    return poblacion[-1], historial

In [9]:
individuos_p1   = 20
generaciones_p1 = 500

poblacion_p1 = [creaIndividuo(inversiones, capital) for _ in range(individuos_p1)]
mejor_p1, historial_p1 = evoluciona(poblacion_p1, generaciones_p1)

print("=" * 50)
print("  RESULTADO — Implementación Manual (Benítez)")
print("=" * 50)
print(f"{'Bono':<6} {'Cant':>6} {'Inversión':>12} {'Rendimiento':>14}")
print("-" * 42)
for nombre, cant, inv in zip(nombres_bonos, mejor_p1, inversiones):
    print(f"{nombre:<6} {cant:>6} {cant*inv.precio:>12,} {cant*inv.precio*inv.rendim:>14,.2f}")
print("-" * 42)
print(f"{'TOTAL':<6} {'':>6} {capitalInvertido(mejor_p1):>12,} {rendimiento(mejor_p1):>14,.2f}")
print(f"\nCapital sobrante: ${capital - capitalInvertido(mejor_p1):,}")

  RESULTADO — Implementación Manual (Benítez)
Bono     Cant    Inversión    Rendimiento
------------------------------------------
A           5        2,500         375.00
B           0            0           0.00
C           1          300           9.00
D           6        4,800         480.00
E           6        2,400         240.00
------------------------------------------
TOTAL               10,000       1,104.00

Capital sobrante: $0


## Parte 2 - Implementación con DEAP ##

In [11]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'deap', '-q'])

0

In [12]:
import random as rnd
from deap import base, creator, tools, algorithms

# --- FUNCIÓN OBJETIVO DEAP ---
def evaluar(individuo):
    """
    Función de evaluación para DEAP.
    Devuelve tupla (rendimiento,) — DEAP requiere siempre una tupla.
    Si excede el capital, penaliza con rendimiento 0.
    """
    cap_inv = sum(individuo[i] * inversiones[i].precio for i in range(len(inversiones)))
    if cap_inv > capital:
        return (0.0,)   # penalización por restricción
    rend = sum(individuo[i] * inversiones[i].precio * inversiones[i].rendim
               for i in range(len(inversiones)))
    return (rend,)

# --- DEFINICIÓN DE TIPOS (FITNESS Y INDIVIDUO) ---
# Limpiar si ya existe (para re-ejecuciones)
if hasattr(creator, 'FitnessMax'): del creator.FitnessMax
if hasattr(creator, 'Individual'): del creator.Individual

creator.create('FitnessMax', base.Fitness, weights=(1.0,))  # maximizar
creator.create('Individual', list, fitness=creator.FitnessMax)

# --- TOOLBOX: registro de operadores ---
toolbox = base.Toolbox()

# Generador de genes: cantidad aleatoria para cada bono respetando su máximo
def gen_bono(idx):
    return rnd.randint(0, inversiones[idx].cantidad)

# Inicialización del individuo con genes aleatorios por bono
def init_individuo(icls):
    ind = icls([gen_bono(i) for i in range(len(inversiones))])
    # Reparar si excede capital
    while sum(ind[i]*inversiones[i].precio for i in range(len(inversiones))) > capital:
        pos = rnd.randint(0, len(inversiones)-1)
        if ind[pos] > 0:
            ind[pos] -= 1
    return ind

toolbox.register('individual', init_individuo, creator.Individual)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)

# --- OPERADORES ---
# Evaluación
toolbox.register('evaluate', evaluar)

# Selección: torneo de tamaño 3 (selecciona el mejor de 3 individuos al azar)
toolbox.register('select', tools.selTournament, tournsize=3)

# Cruza: cruce de dos puntos
toolbox.register('mate', tools.cxTwoPoint)

# Mutación: entero uniforme respetando límites [0, cantidad_max]
toolbox.register('mutate', tools.mutUniformInt,
                 low=[0]*len(inversiones),
                 up=[inv.cantidad for inv in inversiones],
                 indpb=0.2)

In [13]:
rnd.seed(42)

TAM_POB      = 20
GENERACIONES = 500
PROB_CRUZA   = 0.7
PROB_MUT     = 0.2

# Estadísticas a registrar
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register('max', np.max)
stats.register('avg', np.mean)

# Hall of Fame: guarda el mejor individuo encontrado
hof = tools.HallOfFame(1)

# Población inicial
pop = toolbox.population(n=TAM_POB)

# Ejecutar algoritmo con elitismo (mu+lambda)
pop_final, log = algorithms.eaMuPlusLambda(
    pop, toolbox,
    mu=TAM_POB,
    lambda_=TAM_POB,
    cxpb=PROB_CRUZA,
    mutpb=PROB_MUT,
    ngen=GENERACIONES,
    stats=stats,
    halloffame=hof,
    verbose=False
)

mejor_p2 = hof[0]
historial_p2_max = log.select('max')
historial_p2_avg = log.select('avg')

print("=" * 50)
print("  RESULTADO — DEAP")
print("=" * 50)
cap_inv_p2 = sum(mejor_p2[i]*inversiones[i].precio for i in range(len(inversiones)))
rend_p2    = sum(mejor_p2[i]*inversiones[i].precio*inversiones[i].rendim for i in range(len(inversiones)))

print(f"{'Bono':<6} {'Cant':>6} {'Inversión':>12} {'Rendimiento':>14}")
print("-" * 42)
for nombre, i in zip(nombres_bonos, range(len(inversiones))):
    cant = mejor_p2[i]
    inv  = inversiones[i]
    print(f"{nombre:<6} {cant:>6} {cant*inv.precio:>12,} {cant*inv.precio*inv.rendim:>14,.2f}")
print("-" * 42)
print(f"{'TOTAL':<6} {'':>6} {cap_inv_p2:>12,} {rend_p2:>14,.2f}")
print(f"\nCapital sobrante: ${capital - cap_inv_p2:,}")

  RESULTADO — DEAP
Bono     Cant    Inversión    Rendimiento
------------------------------------------
A           5        2,500         375.00
B           2        1,400          70.00
C           3          900          27.00
D           3        2,400         240.00
E           7        2,800         280.00
------------------------------------------
TOTAL               10,000         992.00

Capital sobrante: $0


In [16]:
rend_manual = rendimiento(mejor_p1)
cap_manual  = capitalInvertido(mejor_p1)

print("\n" + "=" * 58)
print(f"  {'Métrica':<28} {'Manual':>12} {'DEAP':>12}")
print("=" * 58)
for nombre, m1, m2 in zip(nombres_bonos, mejor_p1, mejor_p2):
    print(f"  Bonos {nombre} comprados          {m1:>12}  {m2:>12}")
print("-" * 58)
print(f"  {'Capital invertido':<28} ${cap_manual:>10,}  ${cap_inv_p2:>10,}")
print(f"  {'Capital sobrante':<28} ${capital-cap_manual:>10,}  ${capital-cap_inv_p2:>10,}")
print(f"  {'Rendimiento total':<28} ${rend_manual:>10,.2f}  ${rend_p2:>10,.2f}")
print(f"  {'ROI (%)':<28} {rend_manual/cap_manual*100 if cap_manual>0 else 0:>11.2f}%  "
      f"{rend_p2/cap_inv_p2*100 if cap_inv_p2>0 else 0:>11.2f}%")
print("=" * 58)

ganador = 'Manual' if rend_manual >= rend_p2 else 'DEAP'
print(f"\n Mejor rendimiento en esta ejecución: {ganador}")
print("   (los resultados varían por ser algoritmos estocásticos)")


  Métrica                            Manual         DEAP
  Bonos A comprados                     5             5
  Bonos B comprados                     0             2
  Bonos C comprados                     1             3
  Bonos D comprados                     6             3
  Bonos E comprados                     6             7
----------------------------------------------------------
  Capital invertido            $    10,000  $    10,000
  Capital sobrante             $         0  $         0
  Rendimiento total            $  1,104.00  $    992.00
  ROI (%)                            11.04%         9.92%

 Mejor rendimiento en esta ejecución: Manual
   (los resultados varían por ser algoritmos estocásticos)


### Comparación de Resultados ###

La comparación muestra que la implementación manual usa una
función como `rendimiento(ind)` que devuelve 
un *float* sin penalización directa,
mientras que en DEAP se usa `evaluar(ind)`, 
que retorna una tupla `(float,)` 
e incluye una penalización explícita (rendimiento 0) 
si se excede el capital.


La tabla muestra que en la implementación manual el fitness 
se obtiene directamente como un valor *float* a partir de `rendimiento()` y se usa de forma implícita al ordenar la población con `sort`, sin normalización ni estructura adicional. En cambio, en DEAP el fitness es explícito mediante `creator.FitnessMax` con pesos `(1.0,)` para maximización, se asigna como `individuo.fitness.values` a través de `toolbox.evaluate`, y además permite manejar problemas multiobjetivo.


La tabla compara los operadores genéticos entre la implementación manual y DEAP: en selección, la manual usa un método proporcional al rango con una lista ponderada, mientras que DEAP utiliza selección por torneo eligiendo el mejor de un grupo aleatorio. En cruza, la manual aplica un solo punto de corte donde un segmento de un padre reemplaza al otro, mientras que DEAP usa dos puntos (`cxTwoPoint`) intercambiando la parte central. En mutación, la manual cambia genes con baja probabilidad (1%) usando valores aleatorios, mientras que DEAP emplea `mutUniformInt` con probabilidad por gen (`indpb=0.2`) respetando rangos. Finalmente, en elitismo, la manual agrega explícitamente al mejor individuo, mientras que DEAP lo maneja automáticamente con el esquema `mu+lambda`, conservando a los mejores.
